In [19]:
from elasticsearch import Elasticsearch
from elasticsearch import helpers

In [20]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env into environment

es = Elasticsearch(
    os.environ["ES_URL"],
    basic_auth=(
        os.environ["ES_USERNAME"],
        os.environ["ES_PASSWORD"]
    ),
    ca_certs=os.environ["ES_CA_CERT"]
)


## Prepare data

In [21]:
import pandas as pd
df=pd.read_csv("/home/sonu/Desktop/semantic_search_engine/myntra_products_catalog.csv").loc[:4999]

In [22]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [23]:
df.isnull().sum()

ProductID         0
ProductName       0
ProductBrand      0
Gender            0
Price (INR)       0
NumImages         0
Description       0
PrimaryColor    374
dtype: int64

In [24]:
df.fillna("None",inplace=True)

## Convert to vectors

In [25]:
from sentence_transformers import SentenceTransformer

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 520.25it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
descriptions = df["Description"].tolist()

embeddings = model.encode(
    descriptions,
    batch_size=32,     
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Batches: 100%|██████████| 157/157 [06:08<00:00,  2.35s/it]


In [27]:
def generate_actions(df, embeddings, index_name):
    for i, row in df.iterrows():
        yield {
            "_index": index_name,
            "_id": int(row["ProductID"]),
            "_source": {
                "ProductID": int(row["ProductID"]),
                "ProductName": row["ProductName"],
                "ProductBrand": row["ProductBrand"],
                "Gender": row["Gender"],
                "Price (INR)": int(row["Price (INR)"]),
                "NumImages": int(row["NumImages"]),
                "Description": row["Description"],
                "PrimaryColor": row["PrimaryColor"],
                "DescriptionVector": embeddings[i].tolist()
            }
        }


In [28]:
success, _ = helpers.bulk(
    client=es,
    actions=generate_actions(df, embeddings, "all_products"),
    chunk_size=500,
    request_timeout=120
)

print(f"Indexed {success} documents successfully")


/tmp/ipykernel_36451/566486951.py:1: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  success, _ = helpers.bulk(


Indexed 5000 documents successfully


## embeddings is a numoy array
Its unnecessary memory after indexing has been done, once data is in es python embeddings and short term scratch space can go

In [29]:
del embeddings
